# Classes that own a record

A finished record is read-only: its key promises its contents. Some objects need a directory
they keep writing to: a trainer logs losses and checkpoints over hours, a solver saves
intermediate states. datamate served this with `Directory.__init__`; tensordir offers two
ways:

- `@td.record` turns a regular class into one whose `__init__` arguments are a spec and whose
  `self.record` is a **workspace record** for that spec. The first construction is the build.
- A record type with `mutable = True` is a workspace record you hold yourself.

This notebook walks through the status lifecycle of both, and through `in_place` builds that
readers can watch and that resume after a crash.

In [ ]:
import pickle
import tempfile
from pathlib import Path
from typing import Annotated

import numpy as np

import tensordir as td

root = Path(tempfile.mkdtemp(prefix="tensordir-owners-"))
td.set_root(root)

## `@td.record`

Every named `__init__` argument becomes a field of the generated `Trainer.Spec`, typed by its
annotation and with its default. `Annotated[T, td.field(hashed=False)]` keeps an argument out
of the key; `Annotated[T, td.transient]` keeps it out of the spec altogether (a logger, a
callback). Spec fields become attributes backed by `self.spec`.

In [ ]:
log = []  # what __init__ saw, for this notebook


@td.record
class Trainer:
    def __init__(
        self,
        lr: float = 0.1,
        n_units: int = 4,
        seed: int = 0,
        device: Annotated[str, td.field(hashed=False)] = "cpu",  # in the spec, not in the key
        verbose: Annotated[bool, td.transient] = False,  # not in the spec
    ):
        log.append(("init", self.record.status, "loss" in self.record))
        self.verbose = verbose
        self.weights = np.random.default_rng(seed).normal(size=n_units)
        self.record.initial_weights = self.weights  # written as part of the build

    def train(self, n_steps: int = 20) -> float:
        if "summary" in self.record:  # finished earlier, maybe by another process
            return self.record.summary["final_loss"]
        for _ in range(n_steps):
            self.weights = self.weights * (1 - self.lr)
            self.record.append("loss", [float((self.weights**2).sum())])
        self.record.weights = self.weights
        self.record.summary = {"final_loss": float(self.record.loss[-1])}
        return self.record.summary["final_loss"]


Trainer.Spec()

## Lifecycle

| moment | status | writes through `self.record` |
|---|---|---|
| construction, record not `done` | `running` (the constructor holds the record's lease) | go into the build |
| `__init__` returns | `done` | |
| `__init__` raises | `failed`, traceback stored; the exception propagates | discarded |
| construction, record `done` | stays `done`; `__init__` runs again on it | edits |
| any time after the build | stays `done` | edits; each updates `meta["modified"]` |

In [ ]:
print("before:", Trainer.Record.locate(Trainer.Spec()).status)
trainer = Trainer(verbose=True)
print("during __init__:", log[-1])
print("after:", trainer.record.status, trainer.record.keys(), trainer.record.meta["modified"])

Methods keep writing after the build. Each edit takes the record's lease briefly, re-reads its
manifest, writes the field and stamps `modified`; the status stays `done`.

In [ ]:
final = trainer.train()
print(final, trainer.record.status, trainer.record.keys())
print("modified:", trainer.record.meta["modified"] is not None)

Constructing the same spec again reopens the finished record and runs `__init__` on it without
rebuilding, so `train` can skip finished work, here or in another process.

In [ ]:
again = Trainer()
print(log[-1], again.record == trainer.record, again.train() == final)

## Spec attributes

`trainer.lr` reads the spec. Assigning a hashed field replaces the spec with
`spec.replace(lr=...)`, so later writes go to the record of the new spec. Assigning an unhashed
field keeps the record.

In [ ]:
first_record = trainer.record
trainer.device = "cuda"
print("unhashed change keeps the record:", trainer.record == first_record)
trainer.lr = 0.3
print("hashed change moves on:", trainer.record != first_record, trainer.record.status)
trainer.train()
print(trainer.spec)

The generated record type is `Trainer.Record`, so the query tools of the earlier notebooks
apply.

In [ ]:
Trainer.Record.info(sizes=False, varying_only=True)[["key", "status", "spec.lr"]]

## A failing `__init__`

An exception in `__init__` stores a `failed` record with the traceback and propagates to the
caller. The next construction retries.

In [ ]:
attempts = []


@td.record
class Flaky:
    def __init__(self, threshold: float = 0.5):
        attempts.append(len(attempts))
        if len(attempts) == 1:
            raise RuntimeError("the GPU went away")
        self.record.ok = True


try:
    Flaky()
except RuntimeError as exc:
    record = Flaky.Record.locate(Flaky.Spec())
    print("raised:", exc, "| status:", record.status, "|", record.error["type"])

flaky = Flaky()
print("retried:", flaky.record.status, flaky.record.ok, "| attempts:", len(attempts))

## Options

`@td.record(name=..., root=..., base=..., mutable=..., schema_version=...)` sets the record
type id, a class root, a `Record` base class providing typed fields, `dims`, codecs or
`in_place`, and whether the record stays writable after `__init__`.

In [ ]:
from jaxtyping import Float


class TypedBase(td.Record):
    weights: Float[np.ndarray, "n_units"]
    codecs = {"ndarray": "hdf5"}


@td.record(name="Solver", root="solvers", base=TypedBase, mutable=False)
class Solver:
    def __init__(self, n_units: int = 3):
        self.record.weights = np.ones(n_units)


solver = Solver()
solver.record.tree()
print(solver.record.path.relative_to(root), type(solver.record.weights).__name__)
try:
    solver.record.weights = np.zeros(3)
except td.ReadOnlyError:
    print("mutable=False: read-only once __init__ has built it")

Instances pickle with their spec. Unpickling does not run `__init__`; the record is reopened on
first use.

In [ ]:
n_inits = len(log)
clone = pickle.loads(pickle.dumps(trainer))  # other instance attributes pickle as usual
print(clone.spec == trainer.spec, clone.record == trainer.record, "| __init__ ran again:", len(log) > n_inits)

## Workspace records without the decorator

Give an ordinary record type `mutable = True` and hold it. Without a `build`, `get` creates an
empty `done` record and edits start right away. `Record.at(path, spec)` does the same in a
directory you name.

In [ ]:
class Run(td.Record):
    __record_name__ = "Run"
    mutable = True

    class Spec(td.Spec):
        lr: float = 0.1


run = Run.get({"lr": 0.1})
other_handle = Run.get({"lr": 0.1})  # e.g. in another process
run.losses = np.array([1.0, 0.5])
other_handle.notes = {"comment": "converged"}
run.append("steps", [1, 2])
print(sorted(Run.get({"lr": 0.1}).keys()))  # neither handle lost the other's field
del run.losses
print(sorted(other_handle.keys()), run.status)

Writing a field again replaces it (the last write to a field wins); writing an array over an
appended field replaces its rows and keeps it appendable.

In [ ]:
run.steps = np.array([10, 20, 30])
run.append("steps", [40])
print(run.load("steps"))

## `in_place` builds

By default a build runs in a temporary directory that appears by rename when it finishes, so
nobody sees a half-built record. `in_place = True` builds in the final directory instead:

- readers see fields and `append`ed rows while the build runs (training logs);
- a rebuild after a failure keeps what the failed attempt wrote, so a recipe can resume from
  its last checkpoint.

In [ ]:
crash = {"at": 6}  # simulate a node failure at step 6 on the first attempt


class LongFit(td.Record):
    __record_name__ = "LongFit"
    in_place = True

    class Spec(td.Spec):
        n_steps: int = 10

    def build(self, spec):
        start = int(self.load("step")[-1]) + 1 if "step" in self else 0  # resume
        for step in range(start, spec.n_steps):
            if step == crash["at"]:
                crash["at"] = None
                raise RuntimeError(f"node lost at step {step}")
            self.append("step", [step])
            self.append("loss", [1 / (step + 1)])
            if step == 2:
                reader = LongFit.at(self.path)  # another handle sees the rows written so far
                print("a reader during the build:", reader.status, reader.load("step").tolist())


try:
    LongFit.get()
except td.BuildFailed as exc:
    print("failed:", exc.error["message"], "| rows kept:", LongFit.locate().load("step").tolist())

fit = LongFit.get(retry_failed=True)
print("resumed:", fit.status, fit.load("step").tolist())